# Domain classifier for Nigerian Pidgin and Igbo — Colab

Trains **one model per language** (same code) for the domains **education, health, telecommunications,
finance, law** (+ *other*), builds a dataset of **up to 50,000 tokens per domain**, and makes a
**60-item annotation sheet** (10 per domain + 10 *other*).

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.
Run the cells top to bottom (Shift + Enter). One language per run: about 30–45 minutes with the teacher.
To do the other language, change `LANG` and run everything again (*Runtime → Restart session* first).

## Settings

In [ ]:
LANG = "pcm"            # "pcm" = Nigerian Pidgin, "ibo" = Igbo
USE_TEACHER = True      # zero-shot teacher for extra training labels (needs the T4 GPU)
TEACHER_TEXTS = 5000    # texts the teacher labels (~15-25 min on a T4)
REPO_URL = ""           # after pushing to GitHub, e.g. "https://github.com/<you>/pcm-ibo-domain-classifier"
                        # leave "" to upload pcm-ibo-domain-classifier.zip instead
L = f"--lang {LANG}"
print("Language:", LANG)

## 1. Get the code and install

In [ ]:
import os, shutil, glob
os.chdir("/content") if os.path.isdir("/content") else None
REPO_DIR = "pcm-ibo-domain-classifier"
if not os.path.isdir(REPO_DIR):
    if REPO_URL:
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        try:
            from google.colab import files
            up = files.upload()                                  # pick pcm-ibo-domain-classifier.zip
            shutil.unpack_archive(next(n for n in up if n.endswith(".zip")), ".")
        except ImportError:
            shutil.unpack_archive(REPO_DIR + ".zip", ".")
os.chdir(REPO_DIR)
!pip -q install -r requirements.txt
if USE_TEACHER:
    !pip -q install transformers sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set USE_TEACHER = False or switch to a T4 GPU)")

## 2. Download and format the text (GitHub + Hugging Face), ~5–10 minutes

In [ ]:
!python step1_download.py {L}
!python step2_format.py {L}

## 3. Build the training data (human labels + teacher + keyword rules)

In [ ]:
TEACH = f"--teacher nli --teacher-max {TEACHER_TEXTS}" if USE_TEACHER else ""
!PYTHONUNBUFFERED=1 python clf1_build_dataset.py {L} {TEACH}

## 4. Train and evaluate

In [ ]:
!python clf2_train.py {L} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"reports/{LANG}_clf_report.md", encoding="utf-8").read()))

## 5. Label every text and rebuild the clean corpus with the classifier's domains

In [ ]:
!python clf3_predict.py {L} > /dev/null
!python step3_clean.py {L} --use-classifier > /dev/null
!python step4_validate_schema.py {L}

## 6. Final dataset (up to 50,000 tokens per domain) + 60-item annotation sheet

In [ ]:
!python clf4_select.py {L} > /dev/null
display(Markdown(open(f"reports/{LANG}_final_dataset.md", encoding="utf-8").read()))

## 7. Download the results
`<lang>_results.zip`: final dataset, annotation sheet (`reviews/<lang>_annotation.xlsx`), model, reports and
the training data. Unzip it into your local copy of the repository before pushing (see README).

In [ ]:
!zip -qr {LANG}_results.zip data/final/{LANG}_domain_dataset.jsonl reviews/{LANG}_annotation.xlsx models/{LANG}_domain_clf.joblib reports/{LANG}_* data/classifier/{LANG}_train.jsonl data/classifier/{LANG}_dev.jsonl data/classifier/{LANG}_test.jsonl
try:
    from google.colab import files
    files.download(f"{LANG}_results.zip")
except ImportError:
    print(f"saved {LANG}_results.zip")

---
## Part 2 — after you annotate
Fill in `reviews/<lang>_annotation.xlsx`: **correct** = Y/N for the predicted domain; if N, choose **true_domain**.

If the session ended: set the Settings cell, run it and cell 1, then continue here.

## 8. Upload `<lang>_results.zip` (if the session restarted) and the filled sheet

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    for n in up:
        if n.endswith(".zip"):
            shutil.unpack_archive(n, ".")
    for n in up:
        if n.endswith(".xlsx"):
            os.makedirs("reviews", exist_ok=True)
            shutil.move(n, f"reviews/{LANG}_annotation_filled.xlsx")
    print("uploaded:", list(up))
except ImportError:
    pass

## 9. Score the annotation

In [ ]:
!python clf5_score_annotation.py {L} --sheet reviews/{LANG}_annotation_filled.xlsx > /dev/null
display(Markdown(open(f"reports/{LANG}_annotation_score.md", encoding="utf-8").read()))

## 10. (Optional) Retrain with your annotations and rebuild the final dataset
Half of the annotated items are added to training, the other half stay in the test set.

In [ ]:
if not glob.glob(f"data/formatted/{LANG}_*.jsonl"):
    !python step1_download.py {L}
    !python step2_format.py {L}
!python clf2_train.py {L} --gold reviews/{LANG}_annotation_filled.xlsx --train-on-gold > /dev/null
!python clf3_predict.py {L} > /dev/null
!python step3_clean.py {L} --use-classifier > /dev/null
!python clf4_select.py {L} > /dev/null
display(Markdown(open(f"reports/{LANG}_clf_report.md", encoding="utf-8").read()))
display(Markdown(open(f"reports/{LANG}_final_dataset.md", encoding="utf-8").read()))
!zip -qr {LANG}_results_v2.zip data/final/{LANG}_domain_dataset.jsonl models/{LANG}_domain_clf.joblib reports/{LANG}_* reviews/{LANG}_annotation_filled.xlsx
try:
    from google.colab import files
    files.download(f"{LANG}_results_v2.zip")
except ImportError:
    print("saved")